## Libraries
Importing the necessary libraries for data loading, statistical analysis and stationarity testing.

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
warnings.filterwarnings('ignore')



## Load Data
Loading the model-ready dataset containing 151 months of remittance data from May 2013 to November 2025.

## Note on covid_shock Column
The dataset includes a `covid_shock` binary column (0 = normal, 1 = COVID period). This column is excluded from descriptive statistics and stationarity tests as it is a binary indicator variable — stationarity tests and descriptive stats are only meaningful for continuous time series variables. This column is intended for use by the modelling team as a feature to flag anomalous COVID months.

In [2]:

# Load your data
df = pd.read_csv("output/remittance_2012_2025_model_ready.csv")
df['date'] = pd.to_datetime(df['date'])

cols = ['remittance', 'exchange_rate', 'oil_price', 'dofe_departures', 'dofe_lag3', 'dofe_lag6', 'dofe_lag9']
df.head()

,date,remittance,exchange_rate,oil_price,dofe_departures,dofe_lag3,dofe_lag6,dofe_lag9,covid_shock
0,2013-05-01,40065.5,87.91,99.366633,54818.0,51516.0,34990.0,45417.0,0
1,2013-06-01,45815.2,93.02,99.742667,58937.0,55439.0,54304.0,38297.0,0
2,2013-07-01,46119.4,95.30,105.257905,59707.0,58054.0,57951.0,47067.0,0
3,2013-08-01,41867.3,98.26,108.157636,54134.0,54818.0,51516.0,34990.0,0
4,2013-09-01,46168.5,101.59,108.757730,32607.0,58937.0,55439.0,54304.0,0


## Descriptive Statistics
Summary statistics for all variables including mean, standard deviation, min, max and skewness. This table will be included in the research paper.

In [3]:
# 1. DESCRIPTIVE STATISTICS
desc = df[cols].describe().round(2)
desc.loc['skew'] = df[cols].skew().round(2)
desc.loc['iqr'] = (df[cols].quantile(0.75) - df[cols].quantile(0.25)).round(2)
print(desc)

       remittance  exchange_rate  oil_price  dofe_departures  dofe_lag3  \
count      151.00         151.00     151.00           151.00     151.00   
mean     80975.40         115.59      69.51         51846.31   51576.83   
std      33820.36          13.44      20.96         18565.51   18415.08   
min      34516.81          87.91      21.04             0.00       0.00   
25%      55128.58         103.81      53.60         45548.00   45548.00   
50%      72418.96         114.30      68.20         54818.00   54702.00   
75%     100180.83         128.78      82.26         64644.50   63969.00   
max     201230.00         141.67     116.80         84226.00   84226.00   
skew         1.24           0.25       0.23            -1.16      -1.16   
iqr      45052.25          24.97      28.66         19096.50   18421.00   

       dofe_lag6  dofe_lag9  
count     151.00     151.00  
mean    51085.40   50491.79  
std     18186.35   17930.49  
min         0.00       0.00  
25%     44751.00   44008

In [4]:
# Skewness Interpretation
for col in cols:
    skew = desc[col]['skew']
    if abs(skew) < 0.5:
        interp = "Approximately Normal"
    elif skew > 0.5:
        interp = "Right Skewed"
    else:
        interp = "Left Skewed"
    print(f"{col}: {skew} → {interp}")

remittance: 1.24 → Right Skewed
exchange_rate: 0.25 → Approximately Normal
oil_price: 0.23 → Approximately Normal
dofe_departures: -1.16 → Left Skewed
dofe_lag3: -1.16 → Left Skewed
dofe_lag6: -1.17 → Left Skewed
dofe_lag9: -1.16 → Left Skewed


## ADF Test (Augmented Dickey-Fuller)
Tests whether each variable is stationary or not. If p < 0.05 the variable is stationary. If p > 0.05 the variable is non-stationary and will need differencing before modelling.

In [5]:
# 2. ADF TEST
for col in cols:
    result = adfuller(df[col])
    p = result[1]
    status = "Stationary" if p < 0.05 else "Non-Stationary"
    print(f"{col}: p={p:.4f} → {status}")

remittance: p=0.9991 → Non-Stationary
exchange_rate: p=0.9099 → Non-Stationary
oil_price: p=0.1165 → Non-Stationary
dofe_departures: p=0.0954 → Non-Stationary
dofe_lag3: p=0.0934 → Non-Stationary
dofe_lag6: p=0.0961 → Non-Stationary
dofe_lag9: p=0.0708 → Non-Stationary


## KPSS Test
Confirms the ADF results from the opposite direction. If p > 0.05 the variable is stationary. Both tests together give a more reliable conclusion.

In [6]:
# 3. KPSS TEST
for col in cols:
    result = kpss(df[col], regression='c')
    p = result[1]
    status = "Stationary" if p > 0.05 else "Non-Stationary"
    print(f"{col}: p={p:.4f} → {status}")

remittance: p=0.0100 → Non-Stationary
exchange_rate: p=0.0100 → Non-Stationary
oil_price: p=0.1000 → Stationary
dofe_departures: p=0.1000 → Stationary
dofe_lag3: p=0.1000 → Stationary
dofe_lag6: p=0.1000 → Stationary
dofe_lag9: p=0.1000 → Stationary


In [10]:
# 4. FIRST-ORDER DIFFERENCING
diff_cols = ['remittance', 'exchange_rate', 'oil_price', 'dofe_departures', 'dofe_lag3', 'dofe_lag6', 'dofe_lag9']

df_diff = df.copy()
for col in diff_cols:
    df_diff[f'{col}_diff'] = df_diff[col].diff()
    # Drop the first row (NaN from differencing)
df_diff = df_diff.dropna().reset_index(drop=True)

diff_col_names = [f'{col}_diff' for col in diff_cols]

print("Differencing done!")
print(f"Shape: {df_diff.shape}")
df_diff.head()

Differencing done!
Shape: (150, 16)


,date,remittance,exchange_rate,oil_price,dofe_departures,dofe_lag3,dofe_lag6,dofe_lag9,covid_shock,remittance_diff,exchange_rate_diff,oil_price_diff,dofe_departures_diff,dofe_lag3_diff,dofe_lag6_diff,dofe_lag9_diff
0,2013-06-01,45815.2,93.020000,99.742667,58937.0,55439.0,54304.0,38297.0,0,5749.7,5.110000,0.376033,4119.0,3923.0,19314.0,-7120.0
1,2013-07-01,46119.4,95.300000,105.257905,59707.0,58054.0,57951.0,47067.0,0,304.2,2.280000,5.515238,770.0,2615.0,3647.0,8770.0
2,2013-08-01,41867.3,98.260000,108.157636,54134.0,54818.0,51516.0,34990.0,0,-4252.1,2.960000,2.899731,-5573.0,-3236.0,-6435.0,-12077.0
3,2013-09-01,46168.5,101.590000,108.757730,32607.0,58937.0,55439.0,54304.0,0,4301.2,3.330000,0.600094,-21527.0,4119.0,3923.0,19314.0
4,2013-10-01,46998.1,99.239677,105.427134,40514.0,59707.0,58054.0,57951.0,0,829.6,-2.350323,-3.330596,7907.0,770.0,2615.0,3647.0


In [11]:
# ADF TEST ON DIFFERENCED VARIABLES
diff_col_names = [f'{col}_diff' for col in diff_cols]

print("=== ADF TEST ON DIFFERENCED VARIABLES ===")
for col in diff_col_names:
    result = adfuller(df_diff[col])
    p = result[1]
    status = " Stationary" if p < 0.05 else " Still Non-Stationary"
    print(f"{col}: p={p:.4f} → {status}")

=== ADF TEST ON DIFFERENCED VARIABLES ===
remittance_diff: p=0.0000 →  Stationary
exchange_rate_diff: p=0.0000 →  Stationary
oil_price_diff: p=0.0000 →  Stationary
dofe_departures_diff: p=0.0000 →  Stationary
dofe_lag3_diff: p=0.0000 →  Stationary
dofe_lag6_diff: p=0.0000 →  Stationary
dofe_lag9_diff: p=0.0000 →  Stationary


In [12]:
# KPSS TEST ON DIFFERENCED VARIABLES
print("=== KPSS TEST ON DIFFERENCED VARIABLES ===")
for col in diff_col_names:
    result = kpss(df_diff[col], regression='c')
    p = result[1]
    status = "Stationary" if p > 0.05 else " Still Non-Stationary"
    print(f"{col}: p={p:.4f} → {status}")

=== KPSS TEST ON DIFFERENCED VARIABLES ===
remittance_diff: p=0.1000 → Stationary
exchange_rate_diff: p=0.1000 → Stationary
oil_price_diff: p=0.1000 → Stationary
dofe_departures_diff: p=0.1000 → Stationary
dofe_lag3_diff: p=0.1000 → Stationary
dofe_lag6_diff: p=0.1000 → Stationary
dofe_lag9_diff: p=0.1000 → Stationary


In [13]:
# SAVE DIFFERENCED DATASET
df_diff.to_csv("output/remittance_differenced.csv", index=False)
print("Saved: output/remittance_differenced.csv")
print(f"Columns: {list(df_diff.columns)}")

Saved: output/remittance_differenced.csv
Columns: ['date', 'remittance', 'exchange_rate', 'oil_price', 'dofe_departures', 'dofe_lag3', 'dofe_lag6', 'dofe_lag9', 'covid_shock', 'remittance_diff', 'exchange_rate_diff', 'oil_price_diff', 'dofe_departures_diff', 'dofe_lag3_diff', 'dofe_lag6_diff', 'dofe_lag9_diff']
